# In Search Of — preview what gets sent to Notion

Runs the **real pipeline** (the actual functions from `adzuna_jobs.py` + `In_Search_Of.py`) end to end:

1. **Scrape** Adzuna for local job postings (`fetch_adzuna_jobs`).
2. **Enrich** them (`enrich_adzuna_rows`): age-gate (drop >30-day postings) + resolve the real employer apply URL.
3. **Print** the final rows that *would* be written to Notion — so you can eyeball them before anything is saved.

Nothing is written to Notion (the `update_page` write is stubbed). Run it on your own machine (residential IP) where Adzuna doesn't rate-limit you.

Needs `ADZUNA_APP_ID` / `ADZUNA_APP_KEY` (developer.adzuna.com).

## Setup — load the real pipeline functions

In [3]:
import re, json, time, datetime, requests, os, sys
from urllib.parse import urlparse
from pathlib import Path

CODE_DIR = Path("/Users/couch2coders/Documents/GitHub/newsletters/In Search Of/Code")
NL_DIR   = Path("/Users/couch2coders/Documents/GitHub/newsletters/NewsletterCreation/Code")
for p in (CODE_DIR, NL_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Load the enrich helpers from In_Search_Of.py WITHOUT importing it (its import
# needs CLAUDE_API_KEY / Notion env). We slice the helper+enrich region and exec
# it, stubbing the Notion write so we can watch the logic without saving.
src = (CODE_DIR / "In_Search_Of.py").read_text()
_start = src.find("_ADZUNA_ANY_RE = re.compile")
_slice = src[_start:src.index("def main(")]
def update_page(page_id, properties=None):
    pass  # stub — no Notion write in the preview
ISO = {"re": re, "json": json, "time": time, "datetime": datetime,
       "requests": requests, "urlparse": urlparse, "print": print,
       "update_page": update_page}
exec(compile(_slice, str(CODE_DIR / "In_Search_Of.py"), "exec"), ISO)
print("enrich loaded:", "enrich_adzuna_rows" in ISO, "| MAX age:", ISO["MAX_LISTING_AGE_DAYS"], "days")

try:
    import curl_cffi; print("curl_cffi:", curl_cffi.__version__)
except ImportError:
    print("curl_cffi NOT installed (pip install curl_cffi) — fetches may be blocked")

enrich loaded: True | MAX age: 30 days
curl_cffi: 0.15.0


## Set your Adzuna API keys

`getpass`, so they're entered at runtime and never saved into the notebook file.

In [21]:
import getpass
os.environ["ADZUNA_APP_ID"]  = os.environ.get("ADZUNA_APP_ID")  or getpass.getpass("ADZUNA_APP_ID:  ")
os.environ["ADZUNA_APP_KEY"] = os.environ.get("ADZUNA_APP_KEY") or getpass.getpass("ADZUNA_APP_KEY: ")

# adzuna_jobs reads the keys at import time, so set them on the module too.
import adzuna_jobs
adzuna_jobs.ADZUNA_APP_ID  = os.environ["ADZUNA_APP_ID"]
adzuna_jobs.ADZUNA_APP_KEY = os.environ["ADZUNA_APP_KEY"]
print("keys set:", bool(adzuna_jobs.ADZUNA_APP_ID), bool(adzuna_jobs.ADZUNA_APP_KEY))

keys set: True True


## Step 1 — scrape Adzuna postings

Change `NEWSLETTER` / `LIMIT` as you like. Prints each result's API `created` date (note: this is Adzuna's re-index date, not the true post date — that's why we age-gate on the website date in Step 2).

In [24]:
NEWSLETTER = "East_Cobb_Connect"
LIMIT = 8

# Get the newsletter config (location etc.); fall back to a minimal config.
try:
    from newsletters_config import get_newsletter
    cfg = get_newsletter(NEWSLETTER) or {}
except Exception as e:
    print("(using minimal config —", e, ")")
    cfg = {}
cfg.setdefault("name", NEWSLETTER)
cfg.setdefault("display_area", "Marietta, GA")

scraped = adzuna_jobs.fetch_adzuna_jobs(cfg, limit=LIMIT)
print(f"\nScraped {len(scraped)} Adzuna posting(s).")

    · healthcare-nursing-jobs: 25 result(s)
    · social-work-jobs: 3 result(s)
    · domestic-help-cleaning-jobs: 5 result(s)
    · teaching-jobs: 25 result(s)
    · admin-jobs: 25 result(s)
    · trade-construction-jobs: 25 result(s)
    · part-time: 25 result(s)
    · Mindlance Health                   created=2026-06-21 (0d)
    · Carenest Health Services           created=2026-06-16 (5d)
    · Emory Healthcare/Emory University  created=2026-06-13 (8d)
    · Mill Springs Academy               created=2026-06-20 (1d)
    · Northside Hospital Inc.            created=2026-06-20 (1d)
    · Kimmel and Associates              created=2026-06-21 (0d)
    · Lowe's Companies, Inc.             created=2026-06-21 (0d)
    · LRS Healthcare                     created=2026-06-21 (0d)
  → Adzuna: 8 live posting(s) near Marietta, GA across 7 targeted quer(y/ies)

Scraped 8 Adzuna posting(s).


## Step 2 — enrich (age-gate + resolve real apply URL)

Runs the actual `enrich_adzuna_rows`. Watch the `age-gate:` lines — stale postings (>30 days by their true website date) get marked rejected and removed; the rest get their real employer apply URL resolved.

On your home IP this is fast. If Adzuna rate-limits (429), you'll see `cooling down 60s` retry sweeps.

In [26]:
# Build the pool in the shape enrich expects (give each a synthetic id).
pool = []
for i, r in enumerate(scraped):
    pool.append({
        "notion_page_id": f"row-{i}",
        "employer":         r.get("employer", ""),
        "scraped_snippet":  r.get("scraped_snippet", ""),
        "job_listings_url": r.get("job_listings_url", ""),
        "image_url":        r.get("image_url", ""),
        "city":             r.get("city", ""),
    })

# Snapshot before, so we can show which rows got dropped.
orig_by_id = {r["notion_page_id"]: dict(r) for r in pool}
before_ids = set(orig_by_id)

print("--- enrich_adzuna_rows running ---")
ISO["enrich_adzuna_rows"](pool)
print("--- done ---")

--- enrich_adzuna_rows running ---
    · age-gate: Mindlance Health — 0d old, keep
    · age-gate: Carenest Health Services — 5d old, keep
    ✗ stale (222d old > 30d): Emory Healthcare/Emory University — marking rejected
    · age-gate: Mill Springs Academy — 1d old, keep
    · age-gate: Northside Hospital Inc. — 1d old, keep
    · age-gate: Kimmel and Associates — 0d old, keep
    · age-gate: Lowe's Companies, Inc. — 0d old, keep
    · age-gate: LRS Healthcare — 0d old, keep
    ↳ adzuna drill-down: Mindlance Health (Job Listings URL)
    ✗ unresolved apply link (www.adzuna.com): Carenest Health Services — marking rejected
    ↳ adzuna drill-down: Mill Springs Academy (Job Listings URL)
    ↳ adzuna drill-down: Northside Hospital Inc. (Job Listings URL)
      ⚠ https://constructionjobs.com/job/preconstruction-estima → HTTP 522
    ↳ adzuna drill-down: Kimmel and Associates (Job Listings URL)
    ↳ adzuna drill-down: Lowe's Companies, Inc. (Job Listings URL)
    ↳ adzuna drill-down: L

## Step 3 — what would be sent to Notion

These are the final rows after age-gating + apply-URL resolution. The dropped (stale) ones are listed separately.

In [28]:
kept_ids    = {r["notion_page_id"] for r in pool}
dropped_ids = before_ids - kept_ids

print("=" * 78)
print(f"WOULD SEND {len(pool)} POSTING(S) TO NOTION   |   DROPPED {len(dropped_ids)} stale")
print("=" * 78)
for i, r in enumerate(pool, 1):
    u = r.get("job_listings_url", "")
    tag = "real employer URL" if u and "adzuna." not in urlparse(u).netloc.lower() else "still adzuna.com"
    print(f"\n[{i}] {r.get('employer','?')}   ({r.get('city','')})")
    print(f"    apply : {u[:96]}")
    print(f"            ^ {tag}")
    snip = (r.get("scraped_snippet", "") or "").replace("\n", " ")
    print(f"    blurb source: {snip[:140]}")

if dropped_ids:
    print("\n" + "-" * 78)
    print("DROPPED (stale > 30 days, would be marked rejected — NOT sent):")
    for rid in dropped_ids:
        print("   -", orig_by_id[rid].get("employer", "?"))

print("\n(Reminder: nothing was actually written to Notion — update_page is stubbed.)")

WOULD SEND 6 POSTING(S) TO NOTION   |   DROPPED 2 stale

[1] Mindlance Health   (decatur)
    apply : https://www.alliedtravelcareers.com/job/11961910/surgical-tech-travel-surgical-services-needed-i
            ^ real employer URL
    blurb source: JOB POSTING: Travel Surgical Tech - $1,477 per week in Decatur, GA. Employer: Mindlance Health. Location: Decatur, DeKalb County. Salary: $7

[2] Mill Springs Academy   (alpharetta)
    apply : https://workforcenow.adp.com/mascsr/default/mdf/recruitment/recruitment.html?cid=85d87b5e-5ea6-4
            ^ real employer URL
    blurb source: JOB POSTING: Science Teacher - Middle School. Employer: Mill Springs Academy. Location: Alpharetta, Fulton County. Salary: $60,000-$85,000/y

[3] Northside Hospital Inc.   (tuxedo)
    apply : https://careers-northside.icims.com/jobs/113541/front-office-assistant---northside-women%27s-spe
            ^ real employer URL
    blurb source: JOB POSTING: Front Office Assistant - Northside Women's Specialists At

In [29]:
# === FULL-URL printout — run after Step 2 ===
import datetime
dropped_ids = before_ids - {r["notion_page_id"] for r in pool}
print(f"WOULD SEND {len(pool)} POSTING(S)  |  DROPPED {len(dropped_ids)} stale\n")
for i, r in enumerate(pool, 1):
    src    = orig_by_id[r["notion_page_id"]].get("job_listings_url", "")
    apply  = r.get("job_listings_url", "")
    det    = ISO["_fetch_details"](src)
    posted = ISO["_parse_iso_date"](det.get("date_posted", ""))
    age    = (datetime.date.today() - posted).days if posted else None
    print(f"[{i}] {r.get('employer','?')}  ({r.get('city','')})")
    print(f"    posted: {posted}  ({age}d old)")
    print(f"    SOURCE: {src}")
    print(f"    APPLY : {apply}")
    print()
for rid in dropped_ids:
    print("DROPPED (stale):", orig_by_id[rid].get("employer", "?"))

WOULD SEND 6 POSTING(S)  |  DROPPED 2 stale

[1] Mindlance Health  (decatur)
    posted: 2026-06-21  (0d old)
    SOURCE: https://www.adzuna.com/land/ad/5771710646?se=GOREOaJt8RGrKszqFbIEfQ&utm_medium=api&utm_source=893fec3f&v=4D6F82108978D3901EC14678B01217D68974E493
    APPLY : https://www.alliedtravelcareers.com/job/11961910/surgical-tech-travel-surgical-services-needed-in-decatur-georgia

[2] Mill Springs Academy  (alpharetta)
    posted: 2026-06-20  (1d old)
    SOURCE: https://www.adzuna.com/details/5770504769?utm_medium=api&utm_source=893fec3f
    APPLY : https://workforcenow.adp.com/mascsr/default/mdf/recruitment/recruitment.html?cid=85d87b5e-5ea6-4c83-af8f-86381511f7f5&jobId=539467&utm_source=adzuna&utm_medium=adzuna

[3] Northside Hospital Inc.  (tuxedo)
    posted: 2026-06-20  (1d old)
    SOURCE: https://www.adzuna.com/details/5770569448?utm_medium=api&utm_source=893fec3f
    APPLY : https://careers-northside.icims.com/jobs/113541/front-office-assistant---northside-women%27s

## (Optional) Inspect a single Adzuna posting in detail

Paste any `adzuna.com/details/...` URL to watch the date + apply-URL resolution step by step.

In [31]:
URL = ("https://www.adzuna.com/details/5763134132?se=Gqmtky1s8RGrKszqFbIEfQ"
       "&utm_medium=api&utm_source=893fec3f&v=1EC413E219C5C6F1EE241F3B1AB3C84C8611E059")

details = ISO["_fetch_details"](URL)
posted  = ISO["_parse_iso_date"](details.get("date_posted", ""))
age     = (datetime.date.today() - posted).days if posted else None
print("datePosted :", details.get("date_posted"))
print("age (days) :", age, "->", "DROP" if (age and age > ISO["MAX_LISTING_AGE_DAYS"]) else "keep")
if details.get("land"):
    final_apply, _ = ISO["_resolve_apply_url"](details["land"])
    print("apply URL  :", final_apply[:100])
    print("real site? :", "adzuna." not in urlparse(final_apply).netloc.lower())

datePosted : 2025-11-11T10:53:27
age (days) : 222 -> DROP
      ⚠ https://www.adzuna.com/land/ad/5763134132?aztt=eyJhbGci → HTTP 403
apply URL  : https://www.adzuna.com/land/ad/5763134132?aztt=eyJhbGciOiJIUzI1NiJ9.eyJjaSI6IkhIQzNocUp0OFJHcS1OMmhv
real site? : False
